# 24. 생성 파라미터와 대화 형식

> **제24장** · **이론편 대응: 20.5절(디코딩 전략), 20.6절(대화 형식)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (23장에서 설치한 transformers 사용)
> **다운로드**: GPT-2 (23장에서 받았다면 재사용)

---

## 이 장에서 하는 일

23장에서 모델이 **다음 토큰의 확률분포**를 내놓는 것을 봤다.
그 확률에서 **어떻게 하나를 고를 것인가**가 이 장의 주제다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 탐욕적 선택과 그 문제 | 20.5절 |
| 2 | **온도(Temperature) 검증** ★ | 20.5절 |
| 3 | **Top-k와 Top-p** | 20.5절 |
| 4 | 실제 생성 비교 | 20.5절 |
| 5 | **대화 형식(ChatML) 직접 구성** ★ | 20.6절 |
| 6 | `apply_chat_template` 사용 | 20.6절 |
| 7 | 구조화된 출력 | 20.6절 |

**같은 모델·같은 프롬프트라도 이 설정에 따라 결과가 크게 달라진다.**
실무에서 가장 자주 손대는 부분이기도 하다.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "gpt2"

print("모델 불러오는 중... (23장에서 받았다면 즉시 로드)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
print(f"준비 완료 — {MODEL_NAME}, 장치 {device}")

---

## 1. 탐욕적 선택과 그 문제 — 이론편 20.5절

가장 단순한 방법은 **확률이 가장 높은 것을 고르는 것**이다(greedy).

23장에서 썼던 `do_sample=False`가 이 방식이다. 문제가 무엇인지 직접 확인해 보자.

In [ ]:
import torch

prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt")

print("=" * 60)
print("탐욕적 선택 — 같은 입력을 세 번")
print("=" * 60)

for trial in range(3):
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=25,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0])
    print(f"  {trial+1}회: {text}")
    print()

print("-" * 60)
print("세 번 모두 완전히 같다 — 무작위성이 없기 때문이다.")
print()
print("그리고 같은 표현이 반복되는 것이 보인다.")
print("  '가장 그럴듯한 것'만 고르다 보니 순환에 빠지는 것이다.")
print("  → 이론편 20.5절에서 다룬 탐욕적 선택의 한계")

### 왜 반복이 생기는가

한 번 어떤 문장을 만들면, 그 문맥에서 **또 같은 문장이 가장 그럴듯해진다.**
그러면 다시 같은 것을 고르고... 이 순환이 계속된다.

$$\text{"AI is uncertain."} \to \text{"The future of AI is uncertain."} \to \cdots$$

**해법은 무작위성을 넣는 것**이다. 다만 아무렇게나 뽑으면 문장이 무너지므로,
**얼마나 무작위로 할지**를 조절해야 한다. 그 조절 장치가 2~3절의 내용이다.

---

## 2. 온도 — 이론편 20.5절 값 검증 ★

온도는 소프트맥스 **전에** 로짓을 나누는 값이다.

$$P(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T < 1$: 나누면 값이 커지므로 **차이가 벌어진다** → 확실한 것에 집중
- $T = 1$: 원래 확률 그대로
- $T > 1$: 값이 작아져 **차이가 줄어든다** → 고르게 퍼짐

**이론편 20.5절에서 손으로 계산한 값**을 확인해 보자. 로짓이 $(3.0, 1.0, 0.5, 0.2)$일 때다.

In [ ]:
import numpy as np


def softmax_np(x):
    e = np.exp(x - x.max())
    return e / e.sum()


logits = np.array([3.0, 1.0, 0.5, 0.2])

print("=" * 65)
print("이론편 20.5절 — 온도별 확률분포")
print("=" * 65)
print(f"로짓: {logits}")
print()
print(f"{'온도 T':<10}{'확률분포':<40}{'최대 확률'}")
print("-" * 65)

results = {}
for T in [0.5, 1.0, 2.0]:
    p = softmax_np(logits / T)
    results[T] = p
    print(f"{T:<10}{str(p.round(4)):<40}{p.max():.4f}")

print("-" * 65)
print()
print("이론편 값과 대조")
print(f"  T=0.5: {results[0.5].round(3)}   이론편: (0.972, 0.018, 0.007, 0.004)")
print(f"  T=1.0: {results[1.0].round(3)}   이론편: (0.782, 0.106, 0.064, 0.048)")
print(f"  T=2.0: {results[2.0].round(3)}   이론편: (0.526, 0.194, 0.151, 0.130)")

assert abs(results[0.5][0] - 0.972) < 0.001
assert abs(results[1.0][0] - 0.782) < 0.001
assert abs(results[2.0][0] - 0.526) < 0.001
print()
print("[OK] 이론편 20.5절 손계산과 일치")

In [ ]:
import numpy as np

print("=" * 60)
print("온도를 나누는 과정 — 한 단계씩")
print("=" * 60)

for T in [0.5, 1.0, 2.0]:
    scaled = logits / T
    exp_vals = np.exp(scaled - scaled.max())
    probs = exp_vals / exp_vals.sum()

    print(f"\n[T = {T}]")
    print(f"  로짓 / T   : {scaled.round(3)}")
    print(f"  최대값 뺀 후: {(scaled - scaled.max()).round(3)}")
    print(f"  exp        : {exp_vals.round(4)}")
    print(f"  정규화     : {probs.round(4)}")

print()
print("=" * 60)
print("T가 작으면 왜 집중되는가")
print("=" * 60)
print("  로짓 차이 3.0 - 1.0 = 2.0")
print(f"  T=0.5 → 차이가 {2.0/0.5} 로 커짐 → exp에서 격차가 벌어짐")
print(f"  T=2.0 → 차이가 {2.0/2.0} 로 줄어듦 → exp에서 격차가 좁혀짐")
print()
print("나눗셈 하나가 분포의 뾰족함을 조절한다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 온도별 분포 ---
ax = axes[0]
x = np.arange(4)
width = 0.25
colors = {"0.5": "#1E40AF", "1.0": "#64748B", "2.0": "#EA580C"}
for i, T in enumerate([0.5, 1.0, 2.0]):
    p = softmax_np(logits / T)
    ax.bar(x + (i-1)*width, p, width, label=f"T={T}", color=colors[str(T)])
ax.set_xticks(x)
ax.set_xticklabels([f"토큰{i}\n(로짓 {l})" for i, l in enumerate(logits)], fontsize=8)
ax.set_ylabel("확률")
ax.set_title("온도에 따른 확률분포")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 온도에 따른 최대 확률 변화 ---
ax = axes[1]
temps = np.linspace(0.1, 3.0, 100)
max_probs = [softmax_np(logits / T).max() for T in temps]
entropy = [-(softmax_np(logits/T) * np.log(softmax_np(logits/T) + 1e-12)).sum()
           for T in temps]

ax.plot(temps, max_probs, linewidth=2.5, color="#1E40AF", label="최대 확률")
ax2 = ax.twinx()
ax2.plot(temps, entropy, linewidth=2.5, color="#EA580C",
         linestyle="--", label="엔트로피")
ax.axvline(1.0, color="gray", linestyle=":", linewidth=1.5)
ax.set_xlabel("온도 T")
ax.set_ylabel("최대 확률", color="#1E40AF")
ax2.set_ylabel("엔트로피 (불확실성)", color="#EA580C")
ax.set_title("온도가 높을수록 고르게 퍼진다")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("엔트로피는 이론편 6.4절에서 다룬 '불확실성의 양'이다.")
print("온도가 높을수록 엔트로피가 커진다 = 예측하기 어려워진다.")

---

## 3. Top-k와 Top-p — 이론편 20.5절

온도만으로는 부족한 경우가 있다. 온도를 높이면 **아주 부적절한 토큰까지** 뽑힐 수 있기 때문이다.

어휘가 5만 개라면, 각각 확률이 0.00001이라도 다 합치면 무시할 수 없다.

**해법: 후보를 미리 잘라낸다.**

| 방법 | 자르는 기준 |
|---|---|
| Top-k | **개수** — 상위 k개만 남긴다 |
| Top-p | **누적 확률** — 합이 p가 될 때까지만 |

In [ ]:
import numpy as np

p = softmax_np(logits)

print("=" * 65)
print("Top-k 와 Top-p")
print("=" * 65)
print(f"원본 확률: {p.round(4)}")
print(f"누적 확률: {np.cumsum(p).round(4)}")
print()

# --- Top-k ---
print("[Top-k = 2] 상위 2개만 남기고 나머지는 0")
top_k = p.copy()
top_k[2:] = 0
top_k = top_k / top_k.sum()          # 재정규화
print(f"  자르기 전: {p.round(4)}")
print(f"  자른 후  : {np.where(np.arange(4) < 2, p, 0).round(4)}")
print(f"  재정규화 : {top_k.round(4)}")
print(f"  합: {top_k.sum():.4f}")
print()

# --- Top-p ---
print("[Top-p = 0.9] 누적 확률이 0.9를 넘을 때까지만")
cum = np.cumsum(p)
n_keep = int((cum < 0.9).sum()) + 1
print(f"  누적: {cum.round(4)}")
print(f"  0.9를 넘는 지점: {n_keep}번째")
top_p = p.copy()
top_p[n_keep:] = 0
top_p = top_p / top_p.sum()
print(f"  선택된 개수: {n_keep}개")
print(f"  재정규화 : {top_p.round(4)}")
print()
print("-" * 65)
print("두 방식의 차이")
print("  Top-k: 항상 k개 — 분포가 뾰족하든 평평하든 같은 개수")
print("  Top-p: 상황에 따라 개수가 달라짐 — 확실할 때는 적게, 애매할 때는 많이")

In [ ]:
import numpy as np

print("=" * 65)
print("Top-p 가 상황에 적응하는 예")
print("=" * 65)

cases = {
    "확실한 상황": np.array([5.0, 1.0, 0.5, 0.2, 0.1]),
    "애매한 상황": np.array([1.2, 1.1, 1.0, 0.9, 0.8]),
}

for name, lg in cases.items():
    pr = softmax_np(lg)
    cum = np.cumsum(pr)
    n_p = int((cum < 0.9).sum()) + 1

    print(f"\n[{name}]")
    print(f"  확률   : {pr.round(4)}")
    print(f"  누적   : {cum.round(4)}")
    print(f"  Top-p=0.9 → {n_p}개 선택")
    print(f"  Top-k=3  → 3개 선택 (항상 고정)")

print()
print("-" * 65)
print("확실할 때 Top-k=3 은 불필요한 후보까지 포함한다.")
print("Top-p 는 그때그때 필요한 만큼만 남긴다 — 그래서 더 널리 쓰인다.")
print()
print("실무에서 자주 쓰는 조합")
print("  창의적 글쓰기 : temperature=0.9, top_p=0.95")
print("  사실 기반 답변 : temperature=0.3, top_p=0.9")
print("  코드 생성     : temperature=0.2 또는 탐욕적 선택")

---

## 4. 실제 생성 비교 — 이론편 20.5절

이제 실제 모델로 각 설정을 비교한다. **같은 프롬프트, 같은 시드**로 돌려
설정만 바꾸면 결과가 어떻게 달라지는지 본다.

In [ ]:
import torch

prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt")

configs = [
    ("탐욕적 선택",    dict(do_sample=False)),
    ("T=0.7",         dict(do_sample=True, temperature=0.7)),
    ("T=1.5",         dict(do_sample=True, temperature=1.5)),
    ("Top-k=50",      dict(do_sample=True, top_k=50)),
    ("Top-p=0.9",     dict(do_sample=True, top_p=0.9)),
    ("T=0.7+Top-p=0.9", dict(do_sample=True, temperature=0.7, top_p=0.9)),
]

print("=" * 78)
print("설정별 생성 결과 (같은 시드)")
print("=" * 78)

for name, kwargs in configs:
    torch.manual_seed(42)         # 시드 고정 (이론편 11.6절)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20,
                             pad_token_id=tokenizer.eos_token_id, **kwargs)
    text = tokenizer.decode(out[0]).replace("\n", " ")
    print(f"\n[{name}]")
    print(f"  {text}")

print()
print("=" * 78)
print("관찰")
print("  탐욕적: 반복이 나타나기 쉽다")
print("  T가 높으면: 다양하지만 엉뚱해질 수 있다")
print("  Top-p: 부적절한 후보를 걸러 안정적이다")

In [ ]:
import torch
import numpy as np

print("=" * 60)
print("같은 설정, 다른 시드 — 다양성 확인")
print("=" * 60)

prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt")

print("[탐욕적 선택]")
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    print(f"  시드 {seed}: {tokenizer.decode(out[0])[:70]}")

print()
print("[샘플링 (T=0.9, top_p=0.9)]")
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=12, do_sample=True,
                             temperature=0.9, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    print(f"  시드 {seed}: {tokenizer.decode(out[0])[:70]}")

print()
print("-" * 60)
print("탐욕적 선택은 시드와 무관하게 항상 같다.")
print("샘플링은 매번 다르다 — 재현하려면 시드를 기록해야 한다 (이론편 11.6절).")

---

## 5. 대화 형식(ChatML) 직접 구성 — 이론편 20.6절 ★

지금까지는 문장을 이어 쓰기만 했다. **대화형 모델은 어떻게 다를까.**

이론편 20.6절에서 다뤘듯, 사전학습 모델은 **누가 한 말인지 구분하지 못한다.**
그래서 특수 토큰으로 역할 경계를 표시한다.

```
<|im_start|>system
당신은 도움이 되는 어시스턴트입니다.<|im_end|>
<|im_start|>user
파이썬이 뭐야?<|im_end|>
<|im_start|>assistant
```

**마지막 줄이 열려 있다**는 점이 핵심이다. 모델은 여기부터 자기가 이어 써야 한다는 것을 안다.

먼저 이 형식을 **직접 만들어 본다.**

In [ ]:
def build_chatml(messages, add_generation_prompt=True):
    """ChatML 형식으로 대화를 문자열로 만든다 (이론편 20.6절)

    messages: [{"role": "system"/"user"/"assistant", "content": "..."}, ...]
    """
    parts = []
    for msg in messages:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")

    text = "\n".join(parts)

    if add_generation_prompt:
        # 모델이 이어 쓸 자리를 열어 둔다
        text += "\n<|im_start|>assistant\n"

    return text


messages = [
    {"role": "system", "content": "당신은 친절한 AI 도우미입니다."},
    {"role": "user", "content": "파이썬이 뭐야?"},
    {"role": "assistant", "content": "파이썬은 배우기 쉬운 프로그래밍 언어입니다."},
    {"role": "user", "content": "어디에 쓰여?"},
]

print("=" * 60)
print("ChatML 형식 직접 만들기")
print("=" * 60)
formatted = build_chatml(messages)
print(formatted)
print("=" * 60)
print()
print("구조 확인")
print(f"  발화 수      : {len(messages)}")
print(f"  전체 길이    : {len(formatted)}자")
print(f"  마지막 줄    : {repr(formatted[-30:])}")
print()
print("마지막이 '<|im_start|>assistant\\n' 로 끝난다.")
print("→ 모델은 이 다음부터 자기 차례임을 안다.")

In [ ]:
print("=" * 65)
print("세 역할이 하는 일 (이론편 20.6절)")
print("=" * 65)
print(f"{'역할':<14}{'누가 쓰는가':<20}{'무엇을 담는가'}")
print("-" * 65)
print(f"{'system':<14}{'서비스 개발자':<20}{'지침·성격·제약'}")
print(f"{'user':<14}{'사용자':<20}{'질문·요청'}")
print(f"{'assistant':<14}{'모델':<20}{'응답'}")
print("-" * 65)
print()

# system 메시지의 효과 예시
print("system 메시지를 바꾸면 응답 성격이 달라진다")
print()
examples = [
    "당신은 친절한 AI 도우미입니다.",
    "당신은 간결하게 한 문장으로만 답합니다.",
    "당신은 초등학생에게 설명하듯 쉽게 말합니다.",
]
for i, sys_msg in enumerate(examples, 1):
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": "중력이 뭐야?"}]
    text = build_chatml(msgs)
    print(f"  {i}) {sys_msg}")

print()
print("사용자에게는 보이지 않지만 모델의 문맥 창(이론편 20.4절)에는 항상 들어간다.")
print("→ 매번 다시 보내야 하므로 토큰을 계속 차지한다.")

### 형식이 모델마다 다르다

**중요한 주의사항이다.** ChatML은 하나의 방식일 뿐, 표준이 아니다.

| 모델 계열 | 형식 예시 |
|---|---|
| ChatML 계열 | `<\|im_start\|>user ... <\|im_end\|>` |
| Llama 계열 | `[INST] ... [/INST]` |
| 기타 | 모델마다 다름 |

**모델이 학습된 형식을 그대로 써야 한다.** 형식이 어긋나면 성능이 크게 떨어진다.

다행히 `transformers`가 이를 자동으로 처리해 준다. 6절에서 다룬다.

---

## 6. `apply_chat_template` 사용 — 이론편 20.6절

토크나이저에 **`chat_template`**이 들어 있으면, 형식을 자동으로 맞춰 준다.

```python
text = tokenizer.apply_chat_template(messages, tokenize=False,
                                      add_generation_prompt=True)
```

이 방식을 쓰면 모델을 바꿔도 코드를 고칠 필요가 없다.

In [ ]:
print("=" * 65)
print("chat_template 지원 여부")
print("=" * 65)

has_template = getattr(tokenizer, "chat_template", None) is not None
print(f"{MODEL_NAME}: chat_template = {has_template}")
print()

if has_template:
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    print("자동 변환 결과")
    print(text)
else:
    print(f"{MODEL_NAME} 은 대화형 모델이 아니라 템플릿이 없다.")
    print()
    print("GPT-2는 2019년 모델로, 단순히 문장을 이어 쓰도록 학습되었다.")
    print("대화 능력은 27장에서 다룰 SFT(지도 미세조정)를 거쳐야 생긴다.")
    print()
    print("대화형 모델을 쓰려면 이름에 다음이 붙은 것을 고른다:")
    print("  -instruct, -chat, -it  등")
    print()
    print("이 장에서는 5절에서 만든 build_chatml 로 직접 구성한다.")

print()
print("-" * 65)
print("apply_chat_template 을 쓰는 이유")
print("  1) 모델마다 다른 형식을 자동으로 맞춰 준다")
print("  2) 모델을 바꿔도 코드를 고칠 필요가 없다")
print("  3) 특수 토큰 오타 같은 실수를 막는다")

In [ ]:
print("=" * 65)
print("대화형 모델을 쓸 때의 코드 형태")
print("=" * 65)
print()
print("""from transformers import AutoTokenizer, AutoModelForCausalLM

# 대화형 모델 (이름에 instruct/chat 이 붙은 것)
MODEL = "<대화형 모델 이름>"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

messages = [
    {"role": "system", "content": "당신은 친절한 AI 도우미입니다."},
    {"role": "user", "content": "파이썬이 뭐야?"},
]

# 1) 형식 맞추기 (모델에 맞는 템플릿이 자동 적용됨)
text = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,   # 모델이 이어 쓸 자리를 연다
)

# 2) 토큰화
inputs = tok(text, return_tensors="pt")

# 3) 생성
out = model.generate(**inputs, max_new_tokens=200,
                     temperature=0.7, top_p=0.9, do_sample=True)

# 4) 새로 생성된 부분만 잘라내기
answer = tok.decode(out[0][inputs["input_ids"].shape[1]:],
                    skip_special_tokens=True)
print(answer)""")
print()
print("=" * 65)
print("4번 단계가 중요하다")
print("  generate 의 출력에는 입력 프롬프트가 그대로 포함되어 있다.")
print("  입력 길이만큼 잘라내야 모델이 새로 만든 부분만 얻는다.")
print()
print("  skip_special_tokens=True 는 <|im_end|> 같은 것을 제거한다.")

---

## 7. 구조화된 출력 — 이론편 20.6절

역할 구분이 "누가 말하는가"를 표시하는 것이라면, 한 걸음 더 나아가
**응답 내용 자체를 정해진 형식으로** 만들게 할 수 있다.

이것이 32장에서 다룰 **도구 호출(Function Calling)**의 바탕이다.

In [ ]:
import json

print("=" * 65)
print("구조화된 출력 — 왜 필요한가")
print("=" * 65)
print()
print("[문장으로 답하면]")
print('  "서울의 오늘 날씨는 맑고 기온은 23도입니다."')
print("  → 사람은 읽기 쉽지만, 프로그램이 값을 꺼내기 어렵다")
print()
print("[정해진 형식으로 답하면]")
example = {"city": "서울", "weather": "맑음", "temperature": 23}
print(json.dumps(example, ensure_ascii=False, indent=2))
print("  → 프로그램이 바로 값을 읽어 쓸 수 있다")
print()

print("=" * 65)
print("프롬프트로 형식을 지정하는 방법")
print("=" * 65)

system_prompt = """당신은 날씨 정보를 JSON으로만 답하는 도우미입니다.
반드시 아래 형식만 출력하고, 다른 말은 덧붙이지 마세요.

{"city": "도시명", "weather": "날씨", "temperature": 숫자}"""

msgs = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "서울 날씨 알려줘"},
]

print(build_chatml(msgs))
print()
print("-" * 65)
print("모델이 정해진 형식으로 답하면, 다음처럼 바로 쓸 수 있다:")
print()
print("  response = generate(...)")
print("  data = json.loads(response)")
print("  print(data['temperature'])")

In [ ]:
import json

print("=" * 65)
print("응답 파싱 — 실무에서 필요한 방어 코드")
print("=" * 65)


def parse_json_response(text):
    """모델 응답에서 JSON을 안전하게 꺼낸다"""
    # 1) 코드 블록 표시 제거
    cleaned = text.strip()
    for marker in ["```json", "```"]:
        cleaned = cleaned.replace(marker, "")
    cleaned = cleaned.strip()

    # 2) 앞뒤에 설명이 붙은 경우 중괄호 부분만 추출
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]

    # 3) 파싱 시도
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)


# 실제로 자주 만나는 응답 형태들
test_responses = [
    '{"city": "서울", "temperature": 23}',
    '```json\n{"city": "서울", "temperature": 23}\n```',
    '네, 알려드릴게요.\n{"city": "서울", "temperature": 23}',
    '{"city": "서울", "temperature": }',        # 깨진 JSON
]

print(f"{'입력 형태':<40}{'결과'}")
print("-" * 65)
for resp in test_responses:
    data, err = parse_json_response(resp)
    shown = repr(resp[:36]) + ("..." if len(resp) > 36 else "")
    if data:
        print(f"{shown:<40}성공 {data}")
    else:
        print(f"{shown:<40}실패 ({err[:24]})")

print("-" * 65)
print()
print("모델이 항상 완벽한 형식으로 답하지는 않는다.")
print("  - 앞뒤에 설명을 붙이거나")
print("  - 코드 블록으로 감싸거나")
print("  - 형식이 깨지기도 한다")
print()
print("실무에서는 이런 방어 코드가 필수다.")
print("최근에는 형식을 강제하는 기능(structured output)을 제공하는 API도 있다.")

---

## 8. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **20.5** | **T=0.5 → (0.972, 0.018, ...)** | **일치** ✓ |
| 20.5 | T=1.0 → (0.782, 0.106, ...) | 일치 ✓ |
| 20.5 | T=2.0 → (0.526, 0.194, ...) | 일치 ✓ |
| 20.5 | 탐욕적 선택의 반복 문제 | 실제 확인 ✓ |
| 20.6 | ChatML 역할 구분 | 직접 구성 ✓ |

### 생성 파라미터 정리

| 파라미터 | 하는 일 | 값이 크면 |
|---|---|---|
| `temperature` | 분포의 뾰족함 조절 | 다양하지만 엉뚱해짐 |
| `top_k` | 상위 k개만 후보 | 후보가 많아짐 |
| `top_p` | 누적 p까지만 후보 | 후보가 많아짐 |
| `do_sample` | False면 탐욕적 선택 | — |

**용도별 권장 설정**

| 용도 | 설정 |
|---|---|
| 창의적 글쓰기 | `temperature=0.9, top_p=0.95` |
| 사실 기반 답변 | `temperature=0.3, top_p=0.9` |
| 코드 생성 | `temperature=0.2` 또는 탐욕적 |
| 재현이 필요할 때 | `do_sample=False` |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 온도 | 소프트맥스 **전에** 로짓을 나눈다 |
| Top-p 장점 | 상황에 따라 후보 개수가 달라진다 |
| 시드 | 샘플링을 쓰면 재현을 위해 기록 필요 |
| ChatML | 형식이 **모델마다 다르다** |
| `apply_chat_template` | 모델에 맞는 형식을 자동 적용 |
| 응답 파싱 | 방어 코드 필수 — 항상 완벽하지 않다 |
| 출력 자르기 | `out[0][입력길이:]` 로 새 부분만 |

### 다음 장

**25. API로 LLM 사용하기** — 로컬 모델 대신 API로 LLM을 쓰는 방법을 다룬다.
5절에서 만든 ChatML 형식이 API에서는 어떻게 표현되는지 확인하고,
스트리밍 응답과 비용 계산도 함께 본다.